## BÁO CÁO KỸ THUẬT: PHƯƠNG PHÁP HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH FINE-TUNE (FINE-TUNE ONLY)

## 1. THƯ VIỆN SỬ DỤNG VÀ THÔNG SỐ CẤU HÌNH

### Các thư viện chính
- **transformers & bitsandbytes**: Tải mô hình nền, tokenizer và cấu hình QLoRA 4-bit (NF4, Float16 compute).
- **peft**: Cấu hình và quản lý adapter LoRA.
- **trl (SFTTrainer)**: Quản lý và thực thi quá trình Supervised Fine-Tuning.
- **datasets**: Xử lý, tải trực tiếp dữ liệu từ Hugging Face Hub (`TinPhan2007/vietnam-legal-qa-processed`).
- **rouge_score & tqdm**: Đánh giá chỉ số ROUGE-L và trực quan hóa tiến trình.

### Bảng thông số cấu hình chi tiết

| Giai đoạn | Tham số | Giá trị | Giải thích |
| :--- | :--- | :--- | :--- |
| **Model Base** | `model_name` | `Qwen/Qwen2.5-3B-Instruct` | Mô hình nền 3B tham số, lượng tử hóa 4-bit NF4 (`load_in_4bit = True`, compute dtype `torch.float16`). |
| **LoRA Config** | `r / lora_alpha` | `64 / 64` | LoRA rank và hệ số scaling factor. |
| | `target_modules` | `q, k, v, o, gate, up, down_proj` | Áp dụng LoRA vào toàn bộ các lớp Attention và MLP. |
| **Training** | `max_seq_length` | `2048` | Chiều dài chuỗi đầu vào tối đa khi huấn luyện. |
| | `per_device_train_batch_size` | `4` | Batch size trên mỗi step của GPU. |
| | `gradient_accumulation_steps` | `2` | Tích lũy gradient (Batch size hiệu dụng = 2 × 4 = 8). |
| | `learning_rate / epochs` | `1.5e-4 / 1` | Tốc độ học $2 \times 10^{-4}$ và huấn luyện trong 1 epoch. |
| **Inference** | `BATCH_SIZE` | `16` | Xử lý song song 8 mẫu cùng lúc trên GPU. |
| | `padding_side` | `"left"` | Đặt padding bên trái phục vụ batch inference. |
| | `max_new_tokens` | `512` | Giới hạn độ dài sinh token của câu trả lời. |

---

## 2. CẤU TRÚC DỮ LIỆU ĐẦU VÀO CỦA TẬP TRAIN VÀ TEST

### Phân định các trường dữ liệu sử dụng khi Train và Test

* **Dataset nguồn**: `TinPhan2007/vietnam-legal-qa-processed`
* **Giai đoạn Train (Huấn luyện)**:
  * *Trường sử dụng*: Sử dụng trường `messages` đầy đủ (gồm 3 vai trò: `system`, `user`, `assistant`).
  * *Xử lý*: Định dạng hội thoại chuẩn qua `tokenizer.apply_chat_template(convo, tokenize=False)` để chuyển thành chuỗi văn bản gộp duy nhất.
* **Giai đoạn Test (Đánh giá/Inference)**:
  * *Đầu vào (Input Prompt)*: Trích xuất 2 lượt đầu `messages[:2]` (`system` + `user`) kèm cờ mở sinh `add_generation_prompt=True`.
  * *Đầu ra đối chiếu (Ground Truth)*: Nội dung chuẩn của `messages[2]["content"]` (assistant) để tính toán ROUGE-L.
  * *Trường kiểm tra trích dẫn*: Đối chiếu sự xuất hiện chính xác của `law_id` (ví dụ: *"Điều 96"*) trong chuỗi sinh ra.

---

In [ ]:
!pip uninstall torch torchvision torchaudio unsloth xformers -y

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
pip install rouge-score tqdm -q

In [ ]:
import json
import torch
from datasets import Dataset
from rouge_score import rouge_scorer
from tqdm import tqdm
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer
from unsloth.chat_templates import train_on_responses_only
import os
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def check_token_length_hf(hf_dataset_name, split="train"):
    dataset = load_dataset(hf_dataset_name, split=split)
    
    lengths = []
    for item in dataset:
        text = tokenizer.apply_chat_template(item["messages"], tokenize=False)
        tokens = tokenizer.encode(text)
        lengths.append(len(tokens))
        
    total = len(lengths)
    under_1024 = sum(1 for l in lengths if l <= 1024)
    b_1024_2048 = sum(1 for l in lengths if 1024 < l <= 2048)
    over_2048 = sum(1 for l in lengths if l > 2048)
    
    print(f"\n=== THỐNG KÊ ĐỘ DÀI TOKEN MÔ HÌNH QWEN3-4B ===")
    print(f"HF Dataset: {hf_dataset_name} ({split})")
    print(f"Tổng số mẫu: {total}")
    print(f"Độ dài trung bình: {sum(lengths) / total:.1f} tokens")
    print(f"Độ dài lớn nhất: {max(lengths)} tokens")
    print(f" -  <= 1024 tokens : {under_1024} mẫu ({under_1024 / total * 100:.2f}%)")
    print(f" - 1025-2048 tokens: {b_1024_2048} mẫu ({b_1024_2048 / total * 100:.2f}%)")
    print(f" -  > 2048 tokens  : {over_2048} mẫu ({over_2048 / total * 100:.2f}%)")

HF_REPO = "TinPhan2007/vietnam-legal-qa-processed"
check_token_length_hf(HF_REPO, split="train")
check_token_length_hf(HF_REPO, split="test")

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer

HF_REPO = "TinPhan2007/vietnam-legal-qa-processed"
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def format_prompts(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}
    
file_pred = "ft_predictions_evaluated.json"

try:
    with open(file_pred, "r", encoding="utf-8") as f:
        eval_data = json.load(f).get("details", [])

    total = len(eval_data)
    hit_max_512 = 0
    missing_phantich = 0
    cut_and_missing = 0

    for item in eval_data:
        pred_text = item.get("ft_prediction", "")
        num_tokens = len(tokenizer.encode(pred_text))

        is_truncated = num_tokens >= 510
        has_phantich = "phân tích" in pred_text.lower()

        if is_truncated:
            hit_max_512 += 1
        if not has_phantich:
            missing_phantich += 1
        if is_truncated and not has_phantich:
            cut_and_missing += 1

    print("\n=== PHÂN TÍCH LỖI FORMAT TRÊN FILE KẾT QUẢ DỰ ĐOÁN ===")
    print(f"Tổng số mẫu test: {total}")
    if total > 0:
        print(f"1. Số câu bị đụng trần 512 tokens: {hit_max_512} ({hit_max_512/total*100:.2f}%)")
        print(f"2. Số câu bị thiếu phần 'Phân tích & Hướng dẫn': {missing_phantich} ({missing_phantich/total*100:.2f}%)")
        print(f"3. Số câu thiếu 'Phân tích' TRỰC TIẾP DO BỊ CẮT TRẦN 512 TOKENS: {cut_and_missing} ({cut_and_missing/total*100:.2f}%)")

except FileNotFoundError:
    print(f"Chưa tìm thấy '{file_pred}'. Đoạn phân tích này sẽ chạy sau khi bạn hoàn thành bước Inference và tạo file kết quả.")

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig 

HF_REPO = "TinPhan2007/vietnam-legal-qa-processed"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16, 
    device_map="auto",         
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def format_prompts(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}

train_dataset = load_dataset(HF_REPO, split="train").map(format_prompts, batched=True)
val_dataset = load_dataset(HF_REPO, split="validation").map(format_prompts, batched=True)

steps_per_epoch = max(1, len(train_dataset) // (2 * 4))
eval_interval = max(20, steps_per_epoch // 3)

training_args = SFTConfig(
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=20,
    num_train_epochs=6,
    learning_rate=1.5e-4,
    fp16=True,
    bf16=False,
    logging_steps=20,
    output_dir="qwen_legal_ft_outputs",
    eval_strategy="epoch",
    eval_steps=eval_interval,
    save_strategy="epoch",
    save_steps=eval_interval,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

model.save_pretrained("qwen_legal_lora")
tokenizer.save_pretrained("qwen_legal_lora")
print("Huấn luyện hoàn tất và đã lưu checkpoint tốt nhất!")

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

REPO_ID = "TinPhan2007/qwen-legal-lora"
api = HfApi()

api.create_repo(
    repo_id=REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
    exist_ok=True,
    private=False
)

api.upload_folder(
    folder_path="qwen_legal_lora",
    repo_id=REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
    commit_message="Upload Qwen Legal LoRA adapter and tokenizer"
)

print(f"Đã upload thành công lên: https://huggingface.co/{REPO_ID}")